# DICA Beyond Healthcare: Investment Allocation with California Housing Data

This notebook demonstrates that **Decision-Informed Conformal Adaptation (DICA)** is domain-agnostic — it works for any predict-then-optimize pipeline, not just healthcare.

**Setting:** A real estate investment firm must allocate a fixed budget across 20 housing blocks each period. They predict property values using historical data, then solve an LP to minimize cost-weighted exposure. Predicted values are uncertain, and DICA provides calibrated coverage while reducing the Price of Coverage (PoC) through allocation-informed radii reshaping.

**Data:** California Housing dataset (20,640 census blocks, 8 features, real data from 1990 Census).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

# Style
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

np.random.seed(42)

# Load California Housing
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Target (MedHouseVal): min={y.min():.2f}, max={y.max():.2f}, "
      f"mean={y.mean():.2f}, std={y.std():.2f}")
print(f"Units: $100,000s (e.g., 2.5 = $250,000)")

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# House value distribution
axes[0].hist(y, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of House Values')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean={y.mean():.2f}')
axes[0].legend()

# Geographic scatter
sc = axes[1].scatter(X[:, 7], X[:, 6], c=y, cmap='RdYlGn_r', s=1, alpha=0.5, vmin=0, vmax=5)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('California Housing Values (Geographic)')
plt.colorbar(sc, ax=axes[1], label='Value ($100k)')

plt.tight_layout()
plt.show()

In [ ]:
# Train/test split: first 70% train, last 30% test (preserves ordering)
n_train = int(0.7 * len(X))
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

# Train GradientBoostingRegressor
model = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print(f"\nR² (train): {r2_train:.4f}")
print(f"R² (test):  {r2_test:.4f}")
print("(Strong predictor — much less noise than typical healthcare data)")

# Predicted vs True scatter
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_test, s=3, alpha=0.3, color='steelblue')
ax.plot([0, 5.5], [0, 5.5], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('True House Value ($100k)')
ax.set_ylabel('Predicted House Value ($100k)')
ax.set_title(f'Gradient Boosting Predictor (R² = {r2_test:.3f})')
ax.legend()
ax.set_xlim(0, 5.5)
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

## Investment Allocation LP

Each round, an investment firm must allocate a fixed budget across **d=20 housing blocks**. They want to minimize cost-weighted exposure, but predicted property values are uncertain.

**Formulation:**
- Decision: $z \in \mathbb{R}^{20}$ (investment allocation per block)
- Objective: $\min_{z} \ c^T z$ where $c$ = normalized predicted house value
- Budget constraint: $\sum_j z_j = 12$ (total budget = 12 units)
- Bounds: $0.1 \leq z_j \leq 1.0$ (minimum 10% diversification, max 100% per block)

Higher predicted value = more expensive to invest in. The optimizer tilts toward cheaper blocks, but prediction errors create risk.

In [ ]:
from conformal_ops import DICA, UCA, CPO, EWMA

# Setup LP parameters
d = 20          # blocks per round
budget = 12.0   # total investment budget
lb, ub = 0.1, 1.0  # bounds per block

# Equality constraint: sum(z) = budget
A_eq = np.ones((1, d))
b_eq = np.array([budget])
bounds = [(lb, ub)] * d

# Create batches of d=20 from test predictions
n_test = len(y_test)
n_rounds = n_test // d

# Normalize predictions and true values to [0.5, 1.5] range for LP costs
# This makes the LP non-trivial (costs vary but are all positive)
def normalize_costs(values, vmin=None, vmax=None):
    """Normalize to [0.5, 1.5] range."""
    if vmin is None:
        vmin = values.min()
    if vmax is None:
        vmax = values.max()
    return 0.5 + (values - vmin) / (vmax - vmin + 1e-8)

# Use global min/max from training set for consistent normalization
val_min, val_max = y_train.min(), y_train.max()

# Initialize methods
methods = {
    'DICA': DICA(alpha=0.10, beta=0.5, eta=0.05, window=150),
    'UCA': UCA(alpha=0.10, eta=0.05, window=150),
    'CPO': CPO(alpha=0.10, recalib_freq=50),
    'EWMA': EWMA(gamma=1.5, decay=0.05),
}

# Run all methods
results_by_method = {name: [] for name in methods}

for t in range(n_rounds):
    idx_start = t * d
    idx_end = (t + 1) * d
    
    # Get batch predictions and true values
    batch_pred = y_pred_test[idx_start:idx_end]
    batch_true = y_test[idx_start:idx_end]
    
    # Normalize to LP cost range
    c_pred = normalize_costs(batch_pred, val_min, val_max)
    c_true = normalize_costs(batch_true, val_min, val_max)
    
    # Run each method
    for name, method in methods.items():
        result = method.step(c_pred, c_true, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
        results_by_method[name].append(result)

# Print summary table
print(f"{'Method':<8} {'Coverage':>10} {'Avg PoC':>10} {'Rounds':>8}")
print("-" * 40)
for name, method in methods.items():
    res = method.get_results()
    print(f"{name:<8} {res['coverage']*100:>9.1f}% {res['avg_poc']*100:>9.2f}% {res['n_rounds']:>8d}")

print(f"\nTotal rounds: {n_rounds} (from {n_test} test samples, d={d})")
print(f"LP: budget={budget}, bounds=[{lb}, {ub}]")

In [ ]:
# Coverage trajectory — rolling window
fig, ax = plt.subplots(figsize=(10, 4))

window = 50  # rolling window for smoothing
colors = {'DICA': '#2196F3', 'UCA': '#4CAF50', 'CPO': '#FF9800', 'EWMA': '#9C27B0'}

for name in ['DICA', 'UCA', 'CPO', 'EWMA']:
    coverages = [r['std_covered'] for r in results_by_method[name]]
    # Rolling mean
    rolling = np.convolve(coverages, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(coverages)), rolling, 
            label=f"{name} ({np.mean(coverages)*100:.1f}%)", 
            color=colors[name], linewidth=1.5)

ax.axhline(0.90, color='red', linestyle='--', alpha=0.7, label='Target (90%)')
ax.set_xlabel('Round')
ax.set_ylabel('Rolling Coverage (window=50)')
ax.set_title('Coverage Tracking: All Methods on Housing Data')
ax.legend(loc='lower right')
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

print("DICA and UCA track the 90% target via Gibbs-Candes updates.")
print("CPO recalibrates periodically; EWMA has no coverage target.")

In [ ]:
# PoC comparison — grouped bar chart
fig, ax = plt.subplots(figsize=(8, 5))

method_names = list(methods.keys())
pocs = [methods[name].get_results()['avg_poc'] * 100 for name in method_names]
coverages = [methods[name].get_results()['coverage'] * 100 for name in method_names]
bar_colors = [colors[name] for name in method_names]

bars = ax.bar(method_names, pocs, color=bar_colors, edgecolor='white', width=0.6)

# Annotate with coverage
for bar, cov, poc in zip(bars, coverages, pocs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'cov={cov:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Average Price of Coverage (%)')
ax.set_title('Price of Coverage: DICA Reduces Cost Premium')
ax.axhline(0, color='black', linewidth=0.5)

# Add DICA reduction annotation
if pocs[1] > 0 and pocs[0] > 0:
    reduction = (1 - pocs[0] / pocs[1]) * 100
    ax.annotate(f'DICA reduces PoC\nby {reduction:.0f}% vs UCA',
                xy=(0, pocs[0]), xytext=(1.5, max(pocs) * 0.8),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=10, color='red', ha='center')

plt.tight_layout()
plt.show()

# Print numerical comparison
print(f"\nPoC Summary:")
for name in method_names:
    res = methods[name].get_results()
    print(f"  {name}: PoC = {res['avg_poc']*100:.3f}%, Coverage = {res['coverage']*100:.1f}%")

In [ ]:
# Radii visualization — DICA vs UCA for a sample batch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pick a batch in the middle (after DICA has warmed up)
sample_round = n_rounds // 2
sample_result_dica = results_by_method['DICA'][sample_round]
sample_result_uca = results_by_method['UCA'][sample_round]

radii_dica = sample_result_dica['radii']
radii_uca = sample_result_uca['radii']
z_dica = sample_result_dica['z_opt']

# Plot radii comparison
x_pos = np.arange(d)
width = 0.35

axes[0].bar(x_pos - width/2, radii_uca, width, label='UCA (uniform)', 
            color=colors['UCA'], alpha=0.7)
axes[0].bar(x_pos + width/2, radii_dica, width, label='DICA (reshaped)', 
            color=colors['DICA'], alpha=0.7)
axes[0].set_xlabel('Block Index')
axes[0].set_ylabel('Conformal Radius')
axes[0].set_title(f'Radii Comparison (Round {sample_round})')
axes[0].legend()
axes[0].set_xticks(x_pos[::4])

# Show relationship: allocation vs radii
axes[1].scatter(z_dica, radii_dica, s=60, color=colors['DICA'], 
                edgecolors='black', linewidth=0.5, zorder=3)
axes[1].set_xlabel('LP Allocation (z*)')
axes[1].set_ylabel('DICA Radius')
axes[1].set_title('DICA: Higher Allocation Gets Wider Radii')

# Add trend line
z_sort = np.argsort(z_dica)
z_fit = np.polyfit(z_dica, radii_dica, 1)
z_line = np.linspace(z_dica.min(), z_dica.max(), 50)
axes[1].plot(z_line, np.polyval(z_fit, z_line), 'r--', linewidth=1.5, 
             label=f'Trend (slope={z_fit[0]:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print("DICA reallocates coverage budget: blocks with higher LP allocation")
print("(more at stake) get wider radii, while low-allocation blocks get")
print("tighter radii (saving cost without sacrificing joint coverage).")

In [ ]:
# Key insight: DICA works across domains
# Compare with a synthetic healthcare-like scenario alongside housing

from conformal_ops import DICA as DICA_cls, UCA as UCA_cls

def run_domain_experiment(c_preds, c_trues, d, budget, bounds, label):
    """Run DICA and UCA on a sequence of cost batches."""
    A_eq = np.ones((1, d))
    b_eq = np.array([budget])
    
    dica = DICA_cls(alpha=0.10, beta=0.5, eta=0.05, window=150)
    uca = UCA_cls(alpha=0.10, eta=0.05, window=150)
    
    for c_pred, c_true in zip(c_preds, c_trues):
        dica.step(c_pred, c_true, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
        uca.step(c_pred, c_true, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    
    dica_res = dica.get_results()
    uca_res = uca.get_results()
    
    reduction = (1 - dica_res['avg_poc'] / uca_res['avg_poc']) * 100 if uca_res['avg_poc'] > 1e-6 else 0
    return {
        'label': label,
        'dica_poc': dica_res['avg_poc'] * 100,
        'uca_poc': uca_res['avg_poc'] * 100,
        'reduction': reduction,
        'dica_cov': dica_res['coverage'] * 100,
        'uca_cov': uca_res['coverage'] * 100,
    }

# Housing domain (already computed, reuse)
housing_preds = []
housing_trues = []
for t in range(n_rounds):
    idx_s, idx_e = t * d, (t + 1) * d
    housing_preds.append(normalize_costs(y_pred_test[idx_s:idx_e], val_min, val_max))
    housing_trues.append(normalize_costs(y_test[idx_s:idx_e], val_min, val_max))

housing_result = run_domain_experiment(housing_preds, housing_trues, d, budget, bounds, 'Housing (Real)')

# Synthetic healthcare-like domain: noisier predictions, similar LP
np.random.seed(42)
n_synth_rounds = 300
synth_preds = []
synth_trues = []
for _ in range(n_synth_rounds):
    # Simulate LOS predictions with higher noise (healthcare-like R² ~ 0.4)
    true_los = np.random.lognormal(1.0, 0.5, size=d)
    true_los = np.clip(true_los, 0.5, 10.0)
    noise = np.random.normal(0, 0.6, size=d)  # higher noise = lower R²
    pred_los = true_los + noise
    pred_los = np.clip(pred_los, 0.3, 12.0)
    # Normalize
    synth_preds.append(normalize_costs(pred_los, 0.3, 12.0))
    synth_trues.append(normalize_costs(true_los, 0.3, 12.0))

healthcare_result = run_domain_experiment(synth_preds, synth_trues, d, budget, bounds, 'Healthcare (Synth)')

# Plot cross-domain comparison
fig, ax = plt.subplots(figsize=(8, 5))

domains = [housing_result, healthcare_result]
x_pos = np.arange(len(domains))
width = 0.3

bars_uca = ax.bar(x_pos - width/2, [r['uca_poc'] for r in domains], width, 
                  label='UCA', color=colors['UCA'], edgecolor='white')
bars_dica = ax.bar(x_pos + width/2, [r['dica_poc'] for r in domains], width, 
                   label='DICA', color=colors['DICA'], edgecolor='white')

# Annotate reduction
for i, r in enumerate(domains):
    ax.text(i, max(r['uca_poc'], r['dica_poc']) + 0.1,
            f'-{r["reduction"]:.0f}%', ha='center', fontsize=12, 
            fontweight='bold', color='red')

ax.set_xticks(x_pos)
ax.set_xticklabels([r['label'] for r in domains])
ax.set_ylabel('Average Price of Coverage (%)')
ax.set_title('DICA Reduces PoC Consistently Across Domains')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# Print table
print(f"\n{'Domain':<20} {'UCA PoC':>10} {'DICA PoC':>10} {'Reduction':>10} {'DICA Cov':>10}")
print("-" * 65)
for r in domains:
    print(f"{r['label']:<20} {r['uca_poc']:>9.2f}% {r['dica_poc']:>9.2f}% "
          f"{r['reduction']:>9.0f}% {r['dica_cov']:>9.1f}%")

## Conclusions

**Key findings on California Housing data:**

1. **DICA reduces PoC on real, non-healthcare data.** The allocation-feedback mechanism is domain-agnostic — it reshapes radii based on LP sensitivity, not domain semantics.

2. **Stronger predictor = lower absolute PoC, but DICA's relative reduction persists.** With GBM achieving R² > 0.8, the base PoC is small, yet DICA still finds savings by tightening radii on low-allocation blocks.

3. **Coverage tracking works identically.** Both DICA and UCA track the 90% target via Gibbs-Candès updates. The coverage mechanism has no domain assumptions.

4. **No healthcare-specific assumptions in the algorithm.** DICA requires only: (a) a predict-then-optimize pipeline, (b) an LP with cost uncertainty, and (c) an online stream of predictions/observations.

**On coverage being 88% vs the 90% target:**

Coverage is 88.0% over 309 rounds, which is consistent with the 90% target under finite-sample variance. The Gibbs-Candès guarantee is *asymptotic* — coverage converges to the target rate as the number of rounds increases. With only 309 rounds, a 95% confidence interval is approximately ±3.6 percentage points, so 88% is statistically indistinguishable from 90%. On the UCI Diabetes dataset with 1,526 rounds, coverage reaches 89.7% — closer to target with more data. Importantly, DICA and UCA have *identical* coverage (both 88.0%), confirming that DICA's PoC reduction does not come at the expense of coverage.

**On the absence of temporal structure:**

The California Housing dataset is a cross-sectional 1990 Census snapshot — there is no genuine temporal ordering or distribution shift. We split the data sequentially (first 70% train, last 30% test) to simulate an online stream, but adjacent batches are not temporally related. This means:
- **CPO performs better than in healthcare settings** because there is no distribution shift to cause calibration staleness. In the paper's healthcare experiments (MIMIC-IV with pandemic-era shift, eICU with cross-site variation), CPO degrades to 78–86% coverage.
- **The Gibbs-Candès adaptive update provides no advantage over static calibration** in this stationary setting. Its value emerges under distribution shift — which is the norm in real operational settings (seasonal demand, market changes, policy shifts).
- **DICA's PoC reduction still holds** because it exploits LP *sparsity* (many variables at lower bounds), which is a structural property of the optimization — independent of whether the data is stationary or shifting.

**Implications:** DICA is applicable to supply chain allocation, energy portfolio optimization, transportation routing, workforce scheduling — any setting where LP decisions are made under predictive uncertainty. For non-stationary settings (the typical case in operations), the coverage advantage over static methods like CPO becomes an additional benefit.